# 02 – Siivous ja analyysiotos

**Tehtävät:**
1. Puheille lisätään taustatiedot: kausi, puolue, hallitusasema, ministeri ja puheenvuorotyyppi.
2. Merkitään, mitkä puheet rajataan pois analyysiotoksesta ja miksi.
3. Teksti siivotaan sanavartaloiksi, joista muistio `04_sanasto` muodostaa piirteet.

**Syöte:** `Data/processed/speeches_raw.parquet` (muistio 01)
**Tulokset:**
- `Data/processed/speeches_clean.parquet`: yksi rivi per puhe. Alkuperäinen teksti on raakataulussa; taulut yhdistetään sarakkeella `speech_id`.
- `Data/processed/mp_terms.parquet`: yksi rivi per edustaja × kausi × puolue.

Menetelmä seuraa Simolaa, Niemistä ja Tukiaista (2023) ja Gentzkowia, Shapiroa ja Taddya (2019). Alla oleva taulukko vertaa jokaista vaihetta lähteisiin sivunumeroineen.

**Sivunumerot:** Simola ym. = ACE Discussion Paper 160 (toukokuu 2023, tiedosto `Articles/`), jonka liitteessä B on siivouksen kuvaus; lehtiversion (JHPE 2025) sivut ovat eri. Gentzkow ym. = *Econometrica* 87(4), 1307–1340.

## Lähdetarkistus

| vaihe | Simola ym. | Gentzkow ym. | meillä | poikkeaman peruste |
|---|---|---|---|---|
| vain keskustelun puheenvuorot, menettelypuheet pois | s. 44 | – | kaikki puheenvuorotyypit | aineistossa ei ole menettelypuheenvuoroja; puhemiehen repliikit ovat omassa kentässään (muistio 01) |
| puhemies ja varapuhemiehet pois | s. 45 | s. 1311 (virka-asemalla tunnistetut puhujat) | sama | – |
| Ahvenanmaan edustaja pois | s. 46 | – | sama | – |
| ruotsinkieliset puheet pois | s. 46 (puhetasolla) | – | **virketasolla**, ja pois puheet, joissa yli puolet ruotsia | kaksikielisten puheiden ruotsinkieliset virkkeet paljastaisivat RKP:n |
| ei puoluetta → pois | s. 46 | s. 1311 | sama, lisäksi alle 5 edustajan ryhmät | – |
| puoluetta vaihtanut | puolue valtiopäivävuoden alussa, s. 46 | uusi puolue koko kaudelle, s. 1311 (alaviite) | **eduskuntaryhmä puheen hetkellä** | yksikkö on puhe, ja ryhmä tiedetään jokaiselle puheelle |
| sulkeissa olevat lisäykset pois | s. 47 | s. 1311 | sama (myös hakasulkeet) | – |
| symbolit sanoiksi, muut merkit välilyönneiksi, pienet kirjaimet | s. 47 | s. 1311 | sama | – |
| numerot | säilytetään (s. 47) | – | **poistetaan** | vuosiluvut sekä asia- ja pykälänumerot kertovat ajankohdasta ja asiasta, eivät puolueesta |
| yhdysmerkki | välilyönniksi eli sana jakautuu osiin (s. 47) | poistetaan, osat yhdistyvät (s. 1311) | sana jaetaan osiin (*sote-uudistus* → *sote*, *uudistus*) kuten Simolalla ym. | – |
| täytesanat | lista s. 49 | Snowball-lista, s. 1311–1312 | sama lista (`src/stopwords_fi.txt` = Simolan lista, 229/229 sanaa) | – |
| vartalointi Snowball | s. 47 ("Porter2") | s. 1312 (englannin Porter2) | Snowballin suomen stemmeri (PyStemmer) | – |
| nimet, puolueiden nimet, *arvoisa puhemies*, kuukaudet, muut menettelysanat pois | s. 48 (fraaseina) | s. 1312 (menettelyfraasit) | **sanoina katkomerkillä**: sanaparia ei muodostu poistetun sanan yli | sama vaikutus kuin fraasien poistossa, mutta poimii myös yksittäiset sanat; tunnettu puute: *sosiaalidemokraat-*, *vasemmisto* (muistio 05, ajo 5) |
| sanaparit | s. 10 | s. 1311 | **sanat ja sanaparit** luokittelijassa, vain sanaparit leave-out-mittarissa (muistio 04) | suomen yhdyssanat kantavat merkitystä yksinäänkin; Simola ym. mainitsevat sanat vaihtoehtona (s. 10) |
| esiintymisrajat 100 / 10 vuodessa / 10 puhuja-vuotta | s. 10 | s. 1312 | sama (muistio 04) | – |

**Tekemättä:** Gentzkow ym. tarkistavat, muuttuuko tulos, kun esiintymisrajoja tiukennetaan 10 % (s. 1312). Simola ym. huomauttavat, että sääntöpohjainen vartalointi jakaa suomessa saman sanan kahdeksi vartaloksi (s. 11, alaviite 10); lemmatisointia ei tehdä (tiimin päätös 30.9.), ja rajoitus kirjataan raporttiin.

In [1]:
import re
import sys
from collections import Counter

import numpy as np
import pandas as pd
import Stemmer

sys.path.insert(0, "../src")
import config

RAAKA = "../Data/processed/speeches_raw.parquet"
TULOS = "../Data/processed/speeches_clean.parquet"
TULOS_EDUSTAJAT = "../Data/processed/mp_terms.parquet"

puheet = pd.read_parquet(RAAKA)
print("Puheita:", len(puheet))

Puheita: 133648


## 1. Taustatiedot

Määritelmät ovat tiedostossa `src/config.py`, jotta kaikki muistiot käyttävät samoja sääntöjä.

- **Kausi** alkaa päivänä, jona uusi eduskunta kokoontuu: 28.4.2015, 24.4.2019 ja 12.4.2023.
- **Puolue** on puhujan eduskuntaryhmä puheen hetkellä (`config.PARTY_BY_GROUP`). Simola ym. käyttävät puoluetta valtiopäivävuoden alussa; tässä aineistossa ryhmä tiedetään jokaiselle puheelle, joten käytämme sitä. Siniset (Uusi vaihtoehto, myöhemmin Sininen eduskuntaryhmä) on oma puolueensa kesäkuusta 2017.
- **Hallitusasema** määräytyy hallitusten kokoonpanoista (`config.GOVERNMENTS`). Vaalien jälkeinen toimitusministeriö lasketaan edellisen hallituksen jatkoksi. PS lähti hallituksesta 13.6.2017; Siniset jäivät.
- **Ministeri**: puhuja puhuu ministerinä (kentässä `asema` on sana *ministeri*).

In [2]:
puheet["term"] = puheet["start_time"].apply(config.term_of)
puheet["party"] = puheet["group_code"].map(config.PARTY_BY_GROUP)

hallitukset = puheet["start_time"].apply(config.government_of)      # (nimi, hallituspuolueet)
puheet["government"] = [nimi for nimi, _ in hallitukset]
puheet["in_government"] = [puolue in jasenet if isinstance(puolue, str) else np.nan
                           for puolue, (_, jasenet) in zip(puheet["party"], hallitukset)]

puheet["is_minister"] = puheet["role_fi"].str.contains("ministeri", case=False, na=False)
puheet["speech_type"] = puheet["speech_type_code"].map(config.SPEECH_TYPES).fillna("määrittelemätön")
puheet["mp_name"] = puheet["first_name"].fillna("") + " " + puheet["last_name"].fillna("")

puheet["is_aland"] = [nimi in config.ALAND_MPS for nimi in zip(puheet["first_name"], puheet["last_name"])]

pd.crosstab(puheet["party"].fillna("(ei puoluetta)"), puheet["term"])

term,2015-2019,2019-2023,2023-2027
party,,,
(ei puoluetta),80,467,310
KD,2492,1961,725
KESK,11157,7745,5101
KOK,7581,8572,7880
PS,7295,9420,8398
RKP,1684,1635,1039
SDP,10590,8170,10008
SIN,1684,0,0
VAS,4018,2772,2569


## 2. Tekstin siivous

Tavoite: puhe muutetaan sanavartaloiksi (*stems*) niin, että puolueen voi päätellä vain puheen sisällöstä, ei nimistä tai muodollisuuksista. Vaiheet noudattavat Simolaa ym. (s. 47–48); omat poikkeamamme on **lihavoitu** (ks. lähdetarkistus yllä).

1. **Pikakirjoittajan merkinnät pois:** hakasulkeissa ja sulkeissa olevat lisäykset, esim. `[Puhemies koputtaa]` ja `(Naurua)`.
2. **Ruotsinkieliset virkkeet pois.** Simola ym. poistavat ruotsinkieliset puheet mutta pitävät kaksikieliset. **Me tunnistamme kielen virkkeittäin:** virke on ruotsia, jos siinä on vähintään kaksi ruotsin täytesanaa ja enemmän ruotsin kuin suomen täytesanoja. Yksittäiset ruotsin täytesanat suomenkielisessä virkkeessä poistetaan myös. Näin malli ei voi päätellä, että ruotsinkielinen teksti tarkoittaa RKP:tä.
3. **Nimet pois.** Kaikkien aineiston puhujien etu- ja sukunimet poistetaan taivutusmuotoineen, **mutta vain isolla alkukirjaimella kirjoitettuina**. Näin tavalliset sanat eivät katoa (ks. nimilistat alla).
4. **Puolueiden nimet ja lyhenteet pois** (*kokoomus*, *perussuomalaiset*, *demarit*, *vihreät*, *RKP*, *KD*…; SDP:stä molemmat kirjoitusasut *sosialidemokraat-* ja *sosiaalidemokraat-*, jälkimmäinen yleistyy pöytäkirjoissa 2017) sekä **muodollisuudet**: *arvoisa puhemies*, *edustaja*, *kiitos*, *puheenvuoro*, ruotsinkieliset puhuttelut (*ärade talman*) ja kuukausien nimet.
5. **Symbolit sanoiksi** (€ → euro, % → prosentti, § → pykälä), pienet kirjaimet, numerot ja välimerkit pois.
6. **Täytesanat** (*stop words*) **pois** (NLTK:n suomen lista kuten Simolalla ym., `src/stopwords_fi.txt`).
7. **Vartalointi** (*stemming*) Snowballin suomen stemmerillä (PyStemmer) kuten Simolalla ym. (he kutsuvat sitä nimellä Porter2, joka on Snowballin englannin algoritmin nimi), esim. *hallituksen* → *hallituks*.

Poistetun nimen, puolueen tai muodollisuuden paikalle jätetään katkomerkki `|`, jonka yli ei muodosteta sanapareja (*bigrams*). Täytesanat poistetaan ilman katkomerkkiä, joten sanapari voi muodostua niiden yli, kuten artikkeleissa.

Rajoitus: sääntöpohjainen vartalointi on suomelle karkea ja voi jakaa saman sanan kahdeksi vartaloksi (Simola ym. s. 11, alaviite 10: *kotihoido tuen / tuke*). Lemmatisointia (*lemmatisation*) ei tehdä (tiimin päätös 30.9.2026); rajoitus kirjataan raporttiin.

In [3]:
STEMMERI = Stemmer.Stemmer("finnish")
KATKO = "|"

with open("../src/stopwords_fi.txt", encoding="utf-8") as f:
    TAYTESANAT_FI = set(f.read().split())
with open("../src/stopwords_sv.txt", encoding="utf-8") as f:
    TAYTESANAT_SV = set(f.read().split())

SULKEET = re.compile(r"\[[^\]]*\]|\([^)]*\)")
VIRKERAJA = re.compile(r"(?<=[.!?:;])\s+")
SANA = re.compile(r"[A-Za-zÅÄÖåäöÉéÜüŠšŽž]+(?:-[A-Za-zÅÄÖåäöÉéÜüŠšŽž]+)*|€|%|§|\$|\d+")
SYMBOLIT = {"€": "euro", "%": "prosentti", "§": "pykälä", "$": "dollari"}

# Puolueiden nimet ja lyhenteet (pienillä kirjaimilla, koko sana)
PUOLUEEN_NIMI = re.compile(
    r"^(kokoomu\w*|perussuomalai\w*|persu\w*|keskusta(n|a|an|ssa|sta|lle|lla|lta|laiset|laisten|lais\w*)?"
    r"|keskustala\w*|sdp\w*|demar\w*|sosiaa?lidemokraat\w*|vihreät|vihrei\w*|vasemmistoliit\w*"
    r"|kristillisdemokraat\w*|kristilli\w*|rkp\w*|ruotsalai\w*|siniset|sinisten|sinisille|sinisiä|sinisillä"
    r"|kd|ps|kok|kesk|vas|vihr|r|sin|liik|vkk|lib)$"
)

KUUKAUDET = ["tammikuu", "helmikuu", "maaliskuu", "huhtikuu", "toukokuu", "kesäkuu", "heinäkuu",
             "elokuu", "syyskuu", "lokakuu", "marraskuu", "joulukuu"]
MUODOLLISUUDET = {
    "arvoisa", "puhemies", "puhemiehen", "puhemiestä", "puhemiehelle", "puhemiehenä", "herra", "rouva",
    "edustaja", "edustajan", "edustajat", "edustajien", "edustajalle", "edustajaa", "edustajille",
    "kiitos", "kiitoksia", "kollega", "kollegat", "kollegani",
    "puheenvuoro", "puheenvuoron", "puheenvuoroa", "puheenvuorossa", "puheenvuorossaan", "puheenvuoroja",
    "vastauspuheenvuoro", "vastauspuheenvuoron", "vastauspuheenvuoroja", "ryhmäpuheenvuoro", "ryhmäpuheenvuorossa",
    "talman", "talmannen", "ärade", "värderade", "värderad", "herr", "fru", "tack",
}
MUODOLLISUUDET.update(KUUKAUDET)

# Samat muodollisuudet vartaloina (poimii myös taivutusmuodot)
MUODOLLISUUDET_VARTALOT = set(STEMMERI.stemWords(sorted(MUODOLLISUUDET)))
MUODOLLISUUDET_VARTALOT.update(["puhemie", "edustaj"])
for kuukausi in KUUKAUDET:
    MUODOLLISUUDET_VARTALOT.add(STEMMERI.stemWord(kuukausi + "ssa"))
    MUODOLLISUUDET_VARTALOT.add(STEMMERI.stemWord(kuukausi + "n"))

### Nimilistat

Jokaisesta puhujan nimestä muodostetaan taivutusmuotoja (*Orpo, Orpon, Orpolle…*) ja niiden vartalot. Sitten 30 000 puheen otoksesta lasketaan, kuinka usein kukin vartalo esiintyy pienellä kirjoitettuna eli tavallisena sanana.
- **Tiukka lista:** nimet, jotka eivät ole tavallisia sanoja. Ne poistetaan aina, kun ne on kirjoitettu isolla.
- **Monitulkintainen lista:** nimet, jotka ovat myös tavallisia sanoja, eli esiintyvät pienellä vähintään 10 kertaa ja yli 20 %:n verran isolla kirjoitettujen määrästä (*Rinne*, *Aalto*, *Toimi*, *Koski*). Ne poistetaan vain virkkeen keskeltä, jotta virkkeen alussa oleva tavallinen sana säilyy.

In [4]:
PAATTEET = ["", "n", "a", "ä", "lle", "lla", "llä", "lta", "ltä", "ssa", "ssä", "sta", "stä", "han", "hin",
            "on", "in", "na", "nä", "ksi", "a", "ta", "tä", "ineen", "nsa", "nsä"]

nimivartalot = set()
for sarake in ["first_name", "last_name"]:
    for nimi in puheet[sarake].dropna().unique():
        for osa in nimi.strip().split():
            osa = osa.lower()
            palat = [osa]
            for pala in osa.split("-"):          # esim. Halla-aho -> halla, aho
                if len(pala) >= 3:
                    palat.append(pala)
            for pala in palat:
                for paate in PAATTEET:
                    nimivartalot.add(STEMMERI.stemWord(pala + paate))

pienella = Counter()
isolla = Counter()
otos = puheet["text"].sample(30000, random_state=0)
for teksti in otos:
    for sana in re.findall(r"[A-Za-zÅÄÖåäö]{3,}", teksti):
        vartalo = STEMMERI.stemWord(sana.lower())
        if vartalo in nimivartalot:
            if sana[0].islower():
                pienella[vartalo] += 1
            else:
                isolla[vartalo] += 1

NIMET_MONITULKINTAISET = set()
for vartalo in nimivartalot:
    if pienella[vartalo] >= 10 and pienella[vartalo] > 0.2 * isolla[vartalo]:
        NIMET_MONITULKINTAISET.add(vartalo)
NIMET_TIUKKA = nimivartalot - NIMET_MONITULKINTAISET

print("Tiukka lista:", len(NIMET_TIUKKA), "vartaloa")
print("Monitulkintaiset:", len(NIMET_MONITULKINTAISET), "vartaloa, esim.", sorted(NIMET_MONITULKINTAISET)[:25])

Tiukka lista: 1968 vartaloa
Monitulkintaiset: 88 vartaloa, esim. ['aalto', 'aho', 'aino', 'ala', 'ale', 'alv', 'an', 'ano', 'ant', 'arj', 'ase', 'asel', 'at', 'auto', 'el', 'elo', 'em', 'hal', 'han', 'har', 'hel', 'jonk', 'jyrk', 'kai', 'kajo']


In [5]:
def on_ruotsia(virke):
    """Virke on ruotsia, jos siinä on vähintään 2 ruotsin täytesanaa ja enemmän kuin suomen täytesanoja."""
    sanat = re.findall(r"[a-zåäö]+", virke.lower())
    ruotsi = sum(1 for s in sanat if s in TAYTESANAT_SV)
    suomi = sum(1 for s in sanat if s in TAYTESANAT_FI)
    return ruotsi >= 2 and ruotsi > suomi


def on_nimi(sana, kohta_virkkeessa, vartalo, osien_vartalot):
    """Onko isolla kirjoitettu sana jonkun puhujan nimi?"""
    if not sana[0].isupper():
        return False
    if vartalo in NIMET_TIUKKA or any(v in NIMET_TIUKKA for v in osien_vartalot):
        return True
    # monitulkintaiset nimet vain virkkeen keskeltä
    if kohta_virkkeessa > 0:
        if vartalo in NIMET_MONITULKINTAISET or any(v in NIMET_MONITULKINTAISET for v in osien_vartalot):
            return True
    return False


def siivoa(teksti):
    """Palauttaa (siivottu teksti, sanoja siivouksen jälkeen, suomenkielisiä sanoja, ruotsinkielisiä sanoja)."""
    teksti = SULKEET.sub(" ", teksti)
    tulos = []
    sanoja_fi = 0
    sanoja_sv = 0

    for virke in VIRKERAJA.split(teksti):
        if not virke.strip():
            continue
        if on_ruotsia(virke):
            sanoja_sv += len(virke.split())
            tulos.append(KATKO)
            continue
        sanoja_fi += len(virke.split())

        for kohta, sana in enumerate(SANA.findall(virke)):
            if sana in SYMBOLIT:
                tulos.append(SYMBOLIT[sana])
                continue
            if sana.isdigit():
                continue

            pienet = sana.lower()
            osat = pienet.split("-")
            osien_vartalot = [STEMMERI.stemWord(o) for o in osat]
            vartalo = STEMMERI.stemWord(pienet)

            if on_nimi(sana, kohta, vartalo, osien_vartalot):
                tulos.append(KATKO)
                continue
            onko_puolue = any(PUOLUEEN_NIMI.match(o) for o in osat)
            ruotsin_taytesana = pienet in TAYTESANAT_SV and pienet not in TAYTESANAT_FI
            if onko_puolue or pienet in MUODOLLISUUDET or ruotsin_taytesana:
                tulos.append(KATKO)
                continue

            for osa, osan_vartalo in zip(osat, osien_vartalot):
                if osa in TAYTESANAT_FI or len(osa) < 2:
                    continue
                if osan_vartalo in MUODOLLISUUDET_VARTALOT:
                    tulos.append(KATKO)
                    continue
                tulos.append(osan_vartalo)

    # peräkkäiset katkot yhdeksi, ei katkoa alkuun tai loppuun
    siisti = []
    edellinen = KATKO
    for sana in tulos:
        if sana == KATKO and edellinen == KATKO:
            continue
        siisti.append(sana)
        edellinen = sana
    if siisti and siisti[-1] == KATKO:
        siisti.pop()

    sanoja = sum(1 for s in siisti if s != KATKO)
    return " ".join(siisti), sanoja, sanoja_fi, sanoja_sv

### Esimerkki

Keksitty puhe, jossa on puhuttelu, nimi, puolueen nimi, pikakirjoittajan merkintä, luku, symboli ja ruotsinkielinen virke.

In [6]:
esimerkki = ("Arvoisa herra puhemies! Pääministeri Orpon hallitus leikkaa, ja kokoomuksen linja on väärä. "
             "[Puhemies koputtaa] Edustaja Halla-aho puhui 5 %:n korotuksesta ja 10 €:n maksusta. "
             "Jag tycker att det är viktigt och att vi måste göra det för Finland.")
print(esimerkki)
print()
print(siivoa(esimerkki))

Arvoisa herra puhemies! Pääministeri Orpon hallitus leikkaa, ja kokoomuksen linja on väärä. [Puhemies koputtaa] Edustaja Halla-aho puhui 5 %:n korotuksesta ja 10 €:n maksusta. Jag tycker att det är viktigt och att vi måste göra det för Finland.

('pääminister | hallitus leik | linj väärä | puhui prosentti korotuks euro maksu', 10, 22, 14)


In [7]:
# Oikea puhe aineistosta ennen ja jälkeen
nayte = puheet[puheet["speech_type"] == "vastaus"].sample(1, random_state=3).iloc[0]
print(nayte["mp_name"], "/", nayte["party"], "/", nayte["start_time"].date())
print()
print(nayte["text"][:800])
print()
print(siivoa(nayte["text"])[0][:800])

Timo Heinonen / KOK / 2018-09-27

Arvoisa rouva puhemies! Nostin ulkomaalaisten maanomistusasiat eduskunnan kyselytunnilla esille vuonna 2009, ja silloin minua syytettiin siitä, että pelottelen ja lietson vihaa. [Leena Meri: Kuulostaa tutulta!] Nyt kuitenkin tilanne on toisenlainen. Tarvitsemme tiukennuksia ja rajauksia ulkomaalaisten maanhankintaan. Mutta senkin jälkeen välikäsien kautta yhtiöjärjestelyin voi tapahtua niin, että joku kiinteistö joutuu vääränlaisiin käsiin, joten kysyisin, onko mahdollista tehdä sellaista listausta paikoista, joita ei valtio eivätkä kunnat voi myydä kenellekään. Ne pitää pitää valtion tai kuntien käsissä, jos ne ovat sellaisia, että niissä on strategisia ulottuvuuksia. Myös näitä lukuisia Rajavartioston vanhoja rakennuksia myytiin. Silloin sanottiin, että myydään vaan, ei kukaan sinne hytt

nost ulkomaalaist maanomistusasia eduskun kyselytun es vuon silo syytet pelottel lietso viha kuite tila toisenlain tarvits tiukennuks rajauks ulkomaalaist maanhanki

Siivotaan kaikki puheet.

In [8]:
siivotut = [siivoa(teksti) for teksti in puheet["text"]]
puheet["tokens"] = [s[0] for s in siivotut]
puheet["n_tokens"] = [s[1] for s in siivotut]
puheet["n_words_fi"] = [s[2] for s in siivotut]
puheet["n_words_sv"] = [s[3] for s in siivotut]
yhteensa = puheet["n_words_fi"] + puheet["n_words_sv"]
puheet["share_swedish"] = puheet["n_words_sv"] / yhteensa.replace(0, np.nan)
puheet["n_words_raw"] = [len(t.split()) for t in puheet["text"]]
print("Valmis.")

Valmis.


### Tarkistus: jäikö nimiä tai puolueita?

Lasketaan 40 000 puheen otoksesta, kuinka usein poliitikkojen ja puolueiden vartaloita vielä esiintyy. Jäljelle jääneet osumat ovat tavallisia sanoja: *orpo* (orpo lapsi) ja *rin*, joka on enimmäkseen sanojen *rinnalla* ja *rinta* vartalo. Nimeä *Rinne* ei poisteta virkkeen alusta, koska se on monitulkintaisella listalla.

In [9]:
laskuri = Counter()
for teksti in puheet["tokens"].sample(40000, random_state=0):
    laskuri.update(teksti.split())

for vartalo in ["orpo", "marin", "sipil", "purr", "halla", "rin", "kokoomuks", "perussuomalais", "talm"]:
    print(f"{vartalo:15s} {laskuri[vartalo]}")
print()
print("Yleisimmät sanat siivouksen jälkeen:")
print(laskuri.most_common(30))

orpo            14
marin           1
sipil           0
purr            0
halla           0
rin             646
kokoomuks       0
perussuomalais  0
talm            0

Yleisimmät sanat siivouksen jälkeen:
[('|', 93875), ('myös', 61030), ('suome', 42662), ('sit', 37403), ('kaik', 33051), ('hallituks', 28349), ('tule', 26650), ('aika', 25799), ('hyvä', 24255), ('hallitus', 23603), ('tääl', 22591), ('kyl', 22487), ('hyv', 22344), ('osa', 22143), ('esimerk', 22024), ('jo', 21110), ('asia', 20375), ('vielä', 19943), ('täl', 19854), ('pitä', 18506), ('siel', 18034), ('lisä', 17868), ('voi', 17795), ('tode', 17641), ('tehd', 17378), ('myösk', 17158), ('sama', 16312), ('eli', 16234), ('kuite', 15568), ('palj', 15222)]


## 3. Analyysiotoksen (*analysis sample*) rajaukset

Puheita ei poisteta taulukosta. Jokaisesta rajauksesta tehdään oma sarake (`excl_...`), ja sarake `in_sample` kertoo, onko puhe mukana. Näin rajausten vaikutus näkyy suoraan.

| rajaus | peruste |
|---|---|
| tyhjä puhe | ei tekstiä |
| pääosin ruotsiksi | yli puolet sanoista ruotsinkielisissä virkkeissä (Simola ym. poistavat ruotsinkieliset puheet) |
| Ahvenanmaan edustaja | Simola ym. rajaavat pois: edustaa erillistä maakuntaa ja puhuu enimmäkseen ruotsia |
| ei puoluetta | pienet ja yhden hengen ryhmät sekä puheet ilman ryhmää |
| alle 5 edustajan puolue kaudella | liian vähän edustajia mallinnukseen (ei nyt poista yhtään puhetta) |

In [10]:
puheet["excl_empty"] = puheet["n_words_raw"].fillna(0) == 0
puheet["excl_swedish"] = puheet["share_swedish"].fillna(0) > 0.5
puheet["excl_aland"] = puheet["is_aland"]
puheet["excl_no_party"] = puheet["party"].isna()

edustajia = puheet[~puheet["excl_no_party"]].groupby(["term", "party"])["person_id"].nunique().reset_index()
pienet = edustajia[edustajia["person_id"] < config.MIN_MPS_PER_PARTY_TERM]
pienet_puolueet = set(zip(pienet["term"], pienet["party"]))
puheet["excl_small_party"] = [(k, p) in pienet_puolueet for k, p in zip(puheet["term"], puheet["party"])]

rajaukset = ["excl_empty", "excl_swedish", "excl_aland", "excl_no_party", "excl_small_party"]
puheet["in_sample"] = ~puheet[rajaukset].any(axis=1)

jaljella = pd.Series(True, index=puheet.index)
print(f"{'kaikki puheet':30s} {len(puheet):>8,}")
for rajaus in rajaukset:
    jaljella = jaljella & ~puheet[rajaus]
    print(f"{'jäljellä ' + rajaus:30s} {jaljella.sum():>8,}")

kaikki puheet                   133,648
jäljellä excl_empty             133,638
jäljellä excl_swedish           132,086
jäljellä excl_aland             131,816
jäljellä excl_no_party          130,965
jäljellä excl_small_party       130,965


## 4. Tallennus

Siivottu taulu tallennetaan ilman alkuperäistä tekstiä, joka on jo raakataulussa. Edustajataulussa on yksi rivi per edustaja, kausi ja puolue; kauden aikana puoluetta vaihtanut edustaja saa kaksi riviä ja merkinnän `switched_party_in_term`.

In [11]:
sarakkeet = ["speech_id", "record_id", "start_time", "session_year", "term", "government", "speech_type",
             "item_id", "item_title", "person_id", "mp_name", "party", "group_code", "in_government",
             "is_minister", "role_fi", "n_words_raw", "n_words_fi", "n_words_sv", "share_swedish",
             "n_tokens", "tokens"] + rajaukset + ["in_sample"]
puheet[sarakkeet].to_parquet(TULOS, index=False)

otos = puheet[puheet["in_sample"]]
edustajat = otos.groupby(["term", "person_id", "mp_name", "party"]).agg(
    n_speeches=("speech_id", "size"),
    n_tokens=("n_tokens", "sum"),
    first_speech=("start_time", "min"),
    last_speech=("start_time", "max"),
    share_minister=("is_minister", "mean"),
    share_in_government=("in_government", "mean"),
).reset_index()
puolueita = edustajat.groupby(["term", "person_id"])["party"].transform("nunique")
edustajat["switched_party_in_term"] = puolueita > 1
edustajat.to_parquet(TULOS_EDUSTAJAT, index=False)

print("Analyysiotoksessa", len(otos), "puhetta,", otos["person_id"].nunique(), "edustajaa")
print("Edustajataulussa", len(edustajat), "riviä; puoluetta kauden aikana vaihtaneita:",
      edustajat.loc[edustajat["switched_party_in_term"], "person_id"].nunique())

Analyysiotoksessa 130965 puhetta, 370 edustajaa
Edustajataulussa 652 riviä; puoluetta kauden aikana vaihtaneita: 23
